# Part 1: Dogs vs Cats Under Tile Orders
This notebook trains three image-classification architecture families on Dogs vs Cats.
It evaluates how fixed tile-wise tile permutations affect validation accuracy.
Results are aggregated by tile count and plotted as accuracy vs number of tiles.


## Setup

### Global / External Imports
Import third-party libraries used only for notebook orchestration and display.


most basic imports

In [1]:
from pathlib import Path
import importlib
import os
import sys


### Local Imports
Import project modules. Part 1 orchestration lives in this notebook; reusable training, preprocessing, and evaluation helpers stay in `src`.


prep for local imports

In [2]:
from pathlib import Path
import importlib
import importlib.util

_REQUIRED_PROJECT_FILES = (
    Path('src/__init__.py'),
    Path('src/utils/notebook_setup.py'),
    Path('src/evaluation/experiment_results.py'),
)

_CANDIDATE_PROJECT_ROOTS = (
    Path('/content/drive/MyDrive/MLDS_Final_Project'),
    Path('/content/MLDS_Final_Project'),
    Path('/content/drive/MyDrive/Colab Notebooks/MLDS_Final_Project'),
)


def _safe_resolve(path):
    try:
        return Path(path).resolve()
    except OSError:
        return None


def _safe_exists(path):
    try:
        return Path(path).exists()
    except OSError:
        return False


def _safe_cwd():
    try:
        return Path.cwd().resolve()
    except OSError:
        fallback = Path('/content')
        return fallback if _safe_exists(fallback) else Path.home()


def _safe_glob(path, pattern):
    try:
        return list(path.glob(pattern))
    except OSError:
        return []


def _path_looks_like_project_root(path):
    try:
        return all((path / required).is_file() for required in _REQUIRED_PROJECT_FILES)
    except OSError:
        return False


def _candidate_project_roots():
    current = _safe_cwd()
    candidates = [current, *current.parents, *_CANDIDATE_PROJECT_ROOTS]
    my_drive = Path('/content/drive/MyDrive')
    if _safe_exists(my_drive):
        candidates.extend(_safe_glob(my_drive, 'MLDS_Final_Project'))
        candidates.extend(_safe_glob(my_drive, '*/MLDS_Final_Project'))
    return candidates


def _find_project_root():
    seen = set()
    for candidate in _candidate_project_roots():
        candidate = _safe_resolve(candidate)
        if candidate is None or candidate in seen:
            continue
        seen.add(candidate)
        if _path_looks_like_project_root(candidate):
            return candidate
    return None


def _mount_colab_drive_if_available():
    try:
        from google.colab import drive  # type: ignore
    except ImportError:
        return
    drive.mount('/content/drive', force_remount=True)


_PROJECT_ROOT = _find_project_root()
if _PROJECT_ROOT is None:
    _mount_colab_drive_if_available()
    _PROJECT_ROOT = _find_project_root()

if _PROJECT_ROOT is None:
    raise ModuleNotFoundError(
        'Could not find the full MLDS_Final_Project repo. Expected '
        'src/__init__.py, src/utils/notebook_setup.py, and '
        'src/evaluation/experiment_results.py. In Colab, upload or clone '
        'the full repo, or place it at /content/drive/MyDrive/MLDS_Final_Project.'
    )

_COLAB_UTILS_PATH = _PROJECT_ROOT / 'src' / 'utils' / 'colab.py'
_COLAB_SPEC = importlib.util.spec_from_file_location('_mlds_colab_bootstrap', _COLAB_UTILS_PATH)
if _COLAB_SPEC is None or _COLAB_SPEC.loader is None:
    raise ImportError(f'Could not load Colab bootstrap helpers from {_COLAB_UTILS_PATH}')
_colab_bootstrap = importlib.util.module_from_spec(_COLAB_SPEC)
_COLAB_SPEC.loader.exec_module(_colab_bootstrap)

ROOT = _colab_bootstrap.bootstrap_notebook_runtime(_PROJECT_ROOT, force_remount=False)
notebook_setup = importlib.import_module('src.utils.notebook_setup')
ROOT


Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.12.13
Package versions:
  numpy: 2.0.2
  pandas: 2.2.2
  scipy: 1.16.3
  scikit-learn: 1.6.1
  torch: 2.11.0+cu128
  torchvision: 0.26.0+cu128
  timm: 1.0.27
  pyyaml: 6.0.3
torch.cuda.is_available(): True
CUDA device: NVIDIA L4
Project root: /content/drive/MyDrive/MLDS_Final_Project


PosixPath('/content/drive/MyDrive/MLDS_Final_Project')

make local imports

In [3]:
import pandas as pd
from IPython.core.display import Image
from IPython.display import display

import src.evaluation.experiment_results as experiment_results

experiment_results = importlib.reload(experiment_results)
from src.utils.reproducibility import seed_everything  # noqa: E402


### Setup configs

In [4]:
import src.models.factory as model_factory
import src.training.trainer as trainer_module
import src.utils.plotting as plotting
import src.experiments.part1 as part1_experiments
import src.experiments.enhanced_confidence as enhanced_confidence

model_factory = importlib.reload(model_factory)
trainer_module = importlib.reload(trainer_module)
plotting = importlib.reload(plotting)
part1_experiments = importlib.reload(part1_experiments)
enhanced_confidence = importlib.reload(enhanced_confidence)

load_experiment_samples = experiment_results.load_experiment_samples
stage_configured_colab_data_dir = experiment_results.stage_configured_colab_data_dir
plot_accuracy_vs_tiles = experiment_results.plot_accuracy_vs_tiles
experiment_intermediate_figure_path = experiment_results.experiment_intermediate_figure_path
save_aggregated_accuracy = experiment_results.save_aggregated_accuracy
save_rows = experiment_results.save_rows
train_model_on_tile_permutation_records = part1_experiments.train_model_on_tile_permutation_records
train_unfrozen_pretrained_binary_head_on_tile_permutation_records = part1_experiments.train_unfrozen_pretrained_binary_head_on_tile_permutation_records
evaluate_zero_shot_full_pretrained_head_on_tile_permutation_records = part1_experiments.evaluate_zero_shot_full_pretrained_head_on_tile_permutation_records
UNFROZEN_PRETRAINED_BINARY_HEAD_ABLATION = part1_experiments.UNFROZEN_PRETRAINED_BINARY_HEAD_ABLATION
ZERO_SHOT_FULL_PRETRAINED_HEAD_ABLATION = part1_experiments.ZERO_SHOT_FULL_PRETRAINED_HEAD_ABLATION
plot_tile_permutation_samples = plotting.plot_tile_permutation_samples
run_part1_enhanced_confidence_experiment = enhanced_confidence.run_part1_enhanced_confidence_experiment
enhanced_confidence_output_paths = enhanced_confidence.enhanced_confidence_output_paths
from src.preprocessing.samples import class_counts  # noqa: E402
from src.preprocessing.tile_permutations import build_tile_permutation_records, matrix_to_flat_order, tile_permutation_to_jsonable  # noqa: E402
from src.utils.io import save_csv  # noqa: E402


In [5]:
part1_setup = notebook_setup.setup_part1_config()
configs = part1_setup.configs
configs.num_tile_permutations = 3  # Keep easy, medium, and hard in local sample/table output.
configs.stage_colab_data_to_local_disk = True  # Set False on Colab to read directly from Drive instead of copying to /content.
device = part1_setup.device
output_paths = part1_setup.output_paths


Running on Google Colab, adjusting configs...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Selected device: cuda
Python: 3.12.13
Package versions:
  numpy: 2.0.2
  pandas: 2.2.2
  scipy: 1.16.3
  scikit-learn: 1.6.1
  torch: 2.11.0+cu128
  torchvision: 0.26.0+cu128
  timm: 1.0.27
  pyyaml: 6.0.3
torch.cuda.is_available(): True
CUDA device: NVIDIA L4
Selected device: cuda


In [ ]:
print(configs)


In [7]:
%%time
configs.data_dir = stage_configured_colab_data_dir(configs)
print(f'Active data_dir: {configs.data_dir}')


Starting Colab dataset staging.
  Source data directory: /content/drive/MyDrive/MLDS_Final_Project/data/dogs-vs-cats/train
  Expected source ZIP: /content/drive/MyDrive/MLDS_Final_Project/data/dogs-vs-cats/train.zip
  Local ZIP destination: /content/MLDS_Final_Project/data/dogs-vs-cats/train.zip
  Local extraction directory: /content/MLDS_Final_Project/data/dogs-vs-cats/train
  Source directory labeled images: 25000
  Source ZIP labeled images: 25000
  Existing local labeled images: 0
Copying dataset ZIP from Google Drive to local Colab disk: /content/drive/MyDrive/MLDS_Final_Project/data/dogs-vs-cats/train.zip -> /content/MLDS_Final_Project/data/dogs-vs-cats/train.zip (547.8 MiB)
Finished copying dataset ZIP in 18.09s.
Extracting 25000 labeled images from /content/MLDS_Final_Project/data/dogs-vs-cats/train.zip into /content/MLDS_Final_Project/data/dogs-vs-cats/train.
Finished extracting Colab dataset: 25000 files extracted in 3.75s; 25000 labeled images now available locally.
Finished

Local runs use a balanced 256-image subset and 5 training epochs. With `val_fraction=0.2`, the validation split is large enough for accuracy to move in smaller steps than the old 6-image smoke-test split. In aggregated results, `final_epoch` means the metric from the last epoch, and `best_epoch` means the best validation score observed during training.

### make installations before final external imports

In [8]:
# Dependencies are installed during the setup/bootstrap cell above when running in Colab.
print('Dependency setup is complete.')


Dependency setup is complete.


### final imports (after doing pip install if working on colab)

In [9]:
import json
import random
from IPython.core.display import Image
from IPython.display import display
import pandas as pd
import numpy as np
import torch


## Experiments

### Experiment helpers
Shared helpers are kept in `src/evaluation/experiment_results.py`; notebook-specific helpers stay here.

In [10]:
# Part 1 output paths are prepared by notebook_setup.setup_part1_config().


### Setup Exp

In [11]:
# 4. Reproducibility (important for multiple notebooks
random.seed(configs.seed)
np.random.seed(configs.seed)
torch.manual_seed(configs.seed)
torch.set_num_threads(configs.max_threads)  # avoid contention across notebooks

In [12]:
output_paths = part1_setup.output_paths
output_paths


{'raw_results': '/content/drive/MyDrive/MLDS_Final_Project/outputs/results/part1_raw_results.csv',
 'aggregated_results': '/content/drive/MyDrive/MLDS_Final_Project/outputs/results/part1_aggregated_results.csv',
 'figure': '/content/drive/MyDrive/MLDS_Final_Project/outputs/figures/part1_accuracy_vs_tiles.png',
 'intermediate_figures_dir': '/content/drive/MyDrive/MLDS_Final_Project/outputs/figures/intermediate',
 'tile_permutations': '/content/drive/MyDrive/MLDS_Final_Project/outputs/results/part1_tile_permutations.csv',
 'accuracy_plot': '/content/drive/MyDrive/MLDS_Final_Project/outputs/figures/part1_accuracy_vs_tiles.png'}

### Data Loading
Discover the configured Dogs vs Cats split before training.


In [ ]:
train_samples, validation_samples, _ = load_experiment_samples(config=configs, seed=configs.seed)
print(f'Train samples: {len(train_samples)}')
print(f'Validation samples: {len(validation_samples)}')
print('Train class counts:', class_counts(samples=train_samples))
print('Validation class counts:', class_counts(samples=validation_samples))

### Experiments - Run Baselines
Run the configured baseline grid/model/tile permutation sweep. Re-run this cell to regenerate Part 1 outputs.


In [14]:
# Build and save tile permutation records
tile_permutation_records = build_tile_permutation_records(
    tiles_per_side_values=configs.tiles_per_side_values,
    num_tile_permutations=configs.num_tile_permutations,
    seed=configs.seed,
    include_baseline=True,
)
tile_permutation_rows = [
    record.__dict__ | {'tile_permutation': json.dumps(tile_permutation_to_jsonable(record.tile_permutation))}
    for record in tile_permutation_records
]
save_csv(data=tile_permutation_rows, path=output_paths['tile_permutations'])
print(f"Saved {len(tile_permutation_records)} tile permutation records")

permutation_examples = []
for record in tile_permutation_records:
    grid_label = '1x1 / None' if record.tiles_per_side is None else f'{record.tiles_per_side}x{record.tiles_per_side}'
    flat_order = None if record.tile_permutation is None else matrix_to_flat_order(record.tile_permutation)
    permutation_examples.append(
        {
            'grid': grid_label,
            'num_tiles': 1 if record.tiles_per_side is None else record.tiles_per_side * record.tiles_per_side,
            'difficulty': record.tile_permutation_name,
            'tile_permutation_id': record.tile_permutation_id,
            'flat_order_preview': 'None' if flat_order is None else flat_order[: min(12, len(flat_order))],
        }
    )
display(pd.DataFrame(permutation_examples))

if configs.plot_samples:
    import matplotlib.pyplot as plt

    plot_tile_permutation_samples(
        samples=train_samples,
        tile_permutation_records=tile_permutation_records,
        image_size=configs.image_size,
        samples_per_class=1,
        max_records=sum(1 for record in tile_permutation_records if record.tile_permutation is not None),
    )
    plt.show()


Saved 12 tile permutation records


,grid,num_tiles,difficulty,tile_permutation_id,flat_order_preview
0,1x1 / None,1,easy,1,None
1,1x1 / None,1,medium,2,None
2,1x1 / None,1,hard,3,None
3,4x4,16,easy,1,"[1, 0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]"
4,4x4,16,medium,2,"[6, 5, 7, 4, 9, 10, 11, 8, 13, 14, 2, 12]"
5,4x4,16,hard,3,"[14, 9, 8, 13, 6, 5, 4, 12, 3, 1, 0, 2]"
6,7x7,49,easy,1,"[1, 0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]"
7,7x7,49,medium,2,"[24, 9, 10, 47, 17, 13, 7, 15, 16, 12, 18, 19]"
8,7x7,49,hard,3,"[17, 28, 13, 25, 22, 10, 21, 16, 7, 1, 20, 11]"
9,10x10,100,easy,1,"[1, 0, 2, 3, 4, 5, 7, 6, 8, 9, 10, 11]"


In [15]:
# Setup reproducibility and load data
seed = configs.seed
seed_everything(seed=seed, deterministic=configs.deterministic)
# Data already loaded above
print(f"Loaded {len(train_samples)} train, {len(validation_samples)} val samples")

Loaded 20000 train, 5000 val samples


In [16]:
# Prepare shared result accumulation across model runs
all_rows = []
run_id = configs.config_name

### Train Lightweight Model Trio
Train the configured pretrained trio: MobileNetV3-Small, DeiT-Tiny, and gMLP-S16.

In [ ]:
part1_training_loop_figure_paths = []
for model_name in configs.model_names:
    intermediate_figure_path = experiment_intermediate_figure_path(
        configs.figures_dir,
        configs.part,
        f'accuracy_vs_tiles_{model_name}',
    )
    rows = train_model_on_tile_permutation_records(
        config=configs,
        model_name=model_name,
        run_id=run_id,
        train_samples=train_samples,
        validation_samples=validation_samples,
        tile_permutation_records=tile_permutation_records,
        seed=seed,
        device=device,
        raw_results_output_path=output_paths["raw_results"],
        intermediate_figure_output_path=intermediate_figure_path,
    )
    all_rows.extend(rows)
    print("Completed training", model_name, "with", len(rows), "runs")
    if Path(intermediate_figure_path).exists():
        part1_training_loop_figure_paths.append(intermediate_figure_path)
        print(f'Saved intermediate model plot: {intermediate_figure_path}')


In [ ]:
# Aggregate active configured-model baseline results without displaying plots during training.
raw_results = pd.read_csv(filepath_or_buffer=output_paths['raw_results'])
raw_results = raw_results[raw_results['model_name'].isin(configs.model_names)].copy()
if 'ablation_name' in raw_results.columns:
    baseline_raw_results = raw_results[
        raw_results['ablation_name'].isna() | (raw_results['ablation_name'].astype(str).str.strip() == '')
    ].copy()
else:
    baseline_raw_results = raw_results.copy()

aggregated_results = save_aggregated_accuracy(
    raw_results=baseline_raw_results,
    group_columns=['model_name', 'tiles_per_side', 'num_tiles'],
    output_path=output_paths['aggregated_results'],
)
print(f"Saved active baseline aggregated results: {output_paths['aggregated_results']}")
aggregated_results


### Reviewer-Requested Part 1 Controls
These cells append new Part 1 variants without re-running or replacing the existing frozen-backbone baseline rows.


In [ ]:
def _completed_part1_variant_keys(raw_results_path, ablation_name):
    if not Path(raw_results_path).exists():
        return set()
    raw = pd.read_csv(raw_results_path)
    if 'ablation_name' not in raw.columns:
        return set()
    subset = raw[raw['ablation_name'].astype(str) == str(ablation_name)]
    if 'run_status' in subset.columns:
        subset = subset[subset['run_status'].astype(str) == 'completed']
    return {
        (
            str(row.model_name),
            None if pd.isna(row.tiles_per_side) else int(row.tiles_per_side),
            int(row.tile_permutation_id),
        )
        for row in subset.itertuples(index=False)
    }


def _missing_records_for_variant(raw_results_path, ablation_name, model_name, records):
    completed = _completed_part1_variant_keys(raw_results_path, ablation_name)
    missing = []
    for record in records:
        key = (
            str(model_name),
            None if record.tiles_per_side is None else int(record.tiles_per_side),
            int(record.tile_permutation_id),
        )
        if key not in completed:
            missing.append(record)
    return missing


#### Reviewer audit preflight


In [ ]:
from src.preprocessing.labels import AnimalLabel

assert int(AnimalLabel.CAT) == 0
assert int(AnimalLabel.DOG) == 1
train_paths = {path for path, _ in train_samples}
validation_paths = {path for path, _ in validation_samples}
assert train_paths.isdisjoint(validation_paths)
assert class_counts(samples=validation_samples) == {'cat': 2500, 'dog': 2500} or configs.sample_data

hard_10x10_record = next(
    record
    for record in tile_permutation_records
    if record.tiles_per_side == 10 and record.tile_permutation_name == 'hard'
)
baseline_record = next(record for record in tile_permutation_records if record.tiles_per_side is None)
_, hard_validation_loader = part1_experiments.build_tile_permutation_dataloaders(
    config=configs,
    model_name=configs.model_names[0],
    train_samples=train_samples[: configs.batch_size],
    validation_samples=validation_samples[: configs.batch_size],
    record=hard_10x10_record,
)
_, baseline_validation_loader = part1_experiments.build_tile_permutation_dataloaders(
    config=configs,
    model_name=configs.model_names[0],
    train_samples=train_samples[: configs.batch_size],
    validation_samples=validation_samples[: configs.batch_size],
    record=baseline_record,
)
hard_images, hard_targets = next(iter(hard_validation_loader))
baseline_images, baseline_targets = next(iter(baseline_validation_loader))
assert hard_targets.tolist() == baseline_targets.tolist()
assert float((hard_images - baseline_images).abs().mean().item()) > 0.0
print('Audit checks passed: labels, split separation, validation balance, and hard 10x10 tensor transform.')


#### Unfrozen pretrained binary-head control


In [ ]:
%%time
unfrozen_rows = []
for model_name in configs.model_names:
    missing_records = _missing_records_for_variant(
        output_paths['raw_results'],
        UNFROZEN_PRETRAINED_BINARY_HEAD_ABLATION,
        model_name,
        tile_permutation_records,
    )
    print(f'{model_name}: {len(missing_records)} missing unfrozen run(s)')
    if not missing_records:
        continue
    rows = train_unfrozen_pretrained_binary_head_on_tile_permutation_records(
        config=configs,
        model_name=model_name,
        run_id=run_id,
        train_samples=train_samples,
        validation_samples=validation_samples,
        tile_permutation_records=missing_records,
        seed=seed,
        device=device,
        raw_results_output_path=output_paths['raw_results'],
    )
    unfrozen_rows.extend(rows)
print(f'Appended {len(unfrozen_rows)} unfrozen row(s).')


#### Zero-shot full pretrained-head control


In [ ]:
%%time
zero_shot_rows = []
for model_name in configs.model_names:
    missing_records = _missing_records_for_variant(
        output_paths['raw_results'],
        ZERO_SHOT_FULL_PRETRAINED_HEAD_ABLATION,
        model_name,
        tile_permutation_records,
    )
    print(f'{model_name}: {len(missing_records)} missing zero-shot run(s)')
    if not missing_records:
        continue
    rows = evaluate_zero_shot_full_pretrained_head_on_tile_permutation_records(
        config=configs,
        model_name=model_name,
        run_id=run_id,
        train_samples=train_samples,
        validation_samples=validation_samples,
        tile_permutation_records=missing_records,
        seed=seed,
        device=device,
        raw_results_output_path=output_paths['raw_results'],
    )
    zero_shot_rows.extend(rows)
print(f'Appended {len(zero_shot_rows)} zero-shot row(s).')


#### Combined Part 1 comparison table


In [ ]:
PART1_BASELINE_CONDITION = 'frozen_pretrained_binary_head'
PART1_EXPERIMENT_CONDITIONS = [
    PART1_BASELINE_CONDITION,
    UNFROZEN_PRETRAINED_BINARY_HEAD_ABLATION,
    ZERO_SHOT_FULL_PRETRAINED_HEAD_ABLATION,
]


def _condition_figure_path(condition):
    return str(Path(configs.figures_dir) / f'part1_accuracy_vs_tiles_{condition}.png')


def _model_experiment_figure_path(model_name):
    return experiment_intermediate_figure_path(
        configs.figures_dir,
        configs.part,
        f'accuracy_vs_tiles_{model_name}_experiments',
    )


def _add_part1_experiment_condition(frame):
    frame = frame.copy()
    if 'ablation_name' not in frame.columns:
        frame['ablation_name'] = pd.NA
    ablation = frame['ablation_name'].astype('string').fillna('').str.strip()
    frame['experiment_condition'] = ablation.mask(ablation == '', PART1_BASELINE_CONDITION)
    return frame


part1_all_raw = pd.read_csv(output_paths['raw_results'])
part1_all_raw = _add_part1_experiment_condition(part1_all_raw)
part1_all_raw = part1_all_raw[
    part1_all_raw['model_name'].isin(configs.model_names)
    & part1_all_raw['experiment_condition'].isin(PART1_EXPERIMENT_CONDITIONS)
].copy()

part1_variant_summary = save_aggregated_accuracy(
    raw_results=part1_all_raw,
    group_columns=['experiment_condition', 'model_name', 'tiles_per_side', 'num_tiles'],
    output_path=output_paths['aggregated_results'],
)

part1_final_figure_paths = []
for condition in PART1_EXPERIMENT_CONDITIONS:
    condition_raw = part1_all_raw[part1_all_raw['experiment_condition'] == condition].copy()
    condition_summary = part1_variant_summary[
        part1_variant_summary['experiment_condition'] == condition
    ].copy()
    if condition_summary.empty:
        print(f'Skipping final plot for {condition}: no rows found.')
        continue
    figure_path = _condition_figure_path(condition)
    plot_accuracy_vs_tiles(
        aggregated=condition_summary,
        output_path=figure_path,
        model_column='model_name',
        raw_results=condition_raw,
        title=f'{condition}: Mean Validation Accuracy by Tiling Level',
    )
    part1_final_figure_paths.append(figure_path)
    print(f'Saved final model comparison plot for {condition}: {figure_path}')

part1_intermediate_experiment_figure_paths = []
for model_name in configs.model_names:
    model_raw = part1_all_raw[part1_all_raw['model_name'] == model_name].copy()
    model_summary = part1_variant_summary[part1_variant_summary['model_name'] == model_name].copy()
    if model_summary.empty:
        print(f'Skipping experiment comparison plot for {model_name}: no rows found.')
        continue
    figure_path = _model_experiment_figure_path(model_name)
    plot_accuracy_vs_tiles(
        aggregated=model_summary,
        output_path=figure_path,
        model_column='experiment_condition',
        raw_results=model_raw,
        title=f'Intermediate Model Plot: Experiment Comparison by Tiling Level - {model_name}',
    )
    part1_intermediate_experiment_figure_paths.append(figure_path)
    print(f'Saved intermediate experiment comparison plot for {model_name}: {figure_path}')

display(part1_variant_summary.sort_values(['experiment_condition', 'model_name', 'num_tiles']))


#### Reviewer audit: hardest current baseline row


In [ ]:
baseline_raw = part1_all_raw[part1_all_raw['experiment_condition'] == 'frozen_pretrained_binary_head']
hardest_baseline = baseline_raw.sort_values('val_accuracy').head(10)
display(hardest_baseline[[
    'model_name',
    'tiles_per_side',
    'tile_permutation_name',
    'val_accuracy',
    'best_val_accuracy',
]])


### Enhanced Confidence Experiment
Run the reviewer-requested MobileNetV3-Small frozen-backbone confidence sweep. The helper reuses completed seed-42 rows from the existing Part 1 outputs and writes separate enhanced CSV/figure files.


In [ ]:
%%time
RUN_ENHANCED_PART1 = True
part1_enhanced_output_paths = enhanced_confidence_output_paths(
    configs.results_dir,
    configs.figures_dir,
    'part1',
)

if RUN_ENHANCED_PART1:
    part1_enhanced_results = run_part1_enhanced_confidence_experiment(configs, device=device)
    display(part1_enhanced_results)
else:
    enhanced_aggregated_path = Path(part1_enhanced_output_paths['aggregated_results'])
    if enhanced_aggregated_path.exists():
        part1_enhanced_results = pd.read_csv(enhanced_aggregated_path)
        display(part1_enhanced_results)
    else:
        print('Enhanced Part 1 results were not found. Set RUN_ENHANCED_PART1 = True to train missing runs.')

for figure_key in ['all_points_figure', 'mean_by_seed_figure']:
    figure_path = Path(part1_enhanced_output_paths[figure_key])
    if figure_path.exists():
        display(Image(filename=str(figure_path)))
    else:
        print(f'Enhanced Part 1 figure not found yet: {figure_path}')


### Experiments - Results Table
Reload the saved aggregated CSV so the report is reproducible from disk. Each row averages all tile permutation scores for the same model and tile count.


In [19]:
saved_results = {
    'raw': pd.read_csv(filepath_or_buffer=output_paths['raw_results']),
    'aggregated': pd.read_csv(filepath_or_buffer=output_paths['aggregated_results']),
}
display(saved_results['aggregated'])

,model_name,tiles_per_side,num_tiles,mean_final_epoch_val_accuracy,std_final_epoch_val_accuracy,mean_best_epoch_val_accuracy,std_best_epoch_val_accuracy,n_runs
0,deit_tiny,4.0,16,0.971467,0.012290,0.972267,0.012307,3
1,deit_tiny,7.0,49,0.970533,0.018309,0.970800,0.018399,3
2,deit_tiny,10.0,100,0.928400,0.052571,0.929733,0.051948,3
3,deit_tiny,NaN,1,0.987467,0.000306,0.987467,0.000306,3
4,mlp_mixer_base,4.0,16,0.958667,0.022504,0.959667,0.021637,3
5,mlp_mixer_base,7.0,49,0.949133,0.040549,0.950000,0.041257,3
6,mlp_mixer_base,10.0,100,0.893333,0.081597,0.894200,0.080987,3
7,mlp_mixer_base,NaN,1,0.984533,0.000115,0.985200,0.000346,3
8,resnet18,4.0,16,0.942333,0.031903,0.942467,0.031906,3
9,resnet18,7.0,49,0.916000,0.065867,0.916867,0.064476,3


### Experiments - Accuracy Plot
Display the saved accuracy-vs-number-of-tiles plot.


In [ ]:
part1_generated_figure_paths = [
    *globals().get('part1_final_figure_paths', []),
    *globals().get('part1_intermediate_experiment_figure_paths', []),
    *globals().get('part1_training_loop_figure_paths', []),
]

seen_figure_paths = set()
for figure_path in part1_generated_figure_paths:
    if figure_path in seen_figure_paths:
        continue
    seen_figure_paths.add(figure_path)
    if Path(figure_path).exists():
        print(f'Displaying saved figure: {figure_path}')
        display(Image(filename=figure_path))
    else:
        print(f'Plot not found yet: {figure_path}')


In [21]:
# Export this saved notebook to PDF. Save the notebook before running this cell,
# because nbconvert reads the on-disk .ipynb file rather than unsaved editor state.
import importlib

import src.utils.notebook_setup as notebook_setup

notebook_setup = importlib.reload(notebook_setup)
notebook_path = ROOT / 'src' / 'notebooks' / 'part1_solution.ipynb'
export_dir = ROOT / 'outputs' / 'notebooks'
notebook_setup.export_notebook_to_pdf(notebook_path, export_dir)


FileNotFoundError: [Errno 2] No such file or directory: 'kpsewhich'